# C9-dimensionality-reduction — Practice p12 — Solution

**(a) Spectral form.**  Multiplying the thin factors gives
$$S=WW^{\mathsf T}=U\Sigma V^{\mathsf T}V\Sigma U^{\mathsf T}=U\Sigma^2U^{\mathsf T}.$$
Here $V^{\mathsf T}V=I_N$ because the $N$ rows of $V^{\mathsf T}$ are orthonormal.  Expanding the diagonal product gives $S=\sum_i\sigma_i^2u_iu_i^{\mathsf T}$.

**(b) Truncated head.**  The same multiplication for $W_r$ uses $V_r^{\mathsf T}V_r=I_r$ and gives
$$S_r=W_rW_r^{\mathsf T}=U_r\Sigma_r^2U_r^{\mathsf T}=\sum_{i\le r}\sigma_i^2u_iu_i^{\mathsf T}.$$

**(c) Error.**  Subtracting (b) from (a) leaves the tail.  Using $\langle A,B\rangle_F=\operatorname{tr}(A^{\mathsf T}B)$,
$$\langle u_iu_i^{\mathsf T},u_ju_j^{\mathsf T}\rangle_F=(u_i^{\mathsf T}u_j)^2.$$
Orthonormality makes this $0$ for $i\ne j$ and $1$ for $i=j$.  Therefore all cross terms vanish and
$$\|S-S_r\|_F^2=\sum_{i>r}(\sigma_i^2)^2=\sum_{i>r}\sigma_i^4.$$
At $r=0$, the same identity reads $\|S\|_F^2=\sum_i\sigma_i^4$.

In [1]:
import os, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists())
os.environ["GENSIM_DATA_DIR"] = str(_root / "reference" / "cache" / "gensim")

import numpy as np
import gensim.downloader

kv = gensim.downloader.load("glove-wiki-gigaword-100")

WORDS = ["reef", "coral", "lagoon", "dolphin", "whale", "shark",
         "dune", "cactus", "oasis", "camel", "sand", "mirage"]
V = np.asarray(kv[WORDS], dtype=np.float64)
W = V / np.sqrt((V * V).sum(axis=1, keepdims=True))
U, s, Vt = np.linalg.svd(W, full_matrices=False)
S = W @ W.T
W4 = U[:, :4] @ np.diag(s[:4]) @ Vt[:4]
S4 = W4 @ W4.T
lhs = float(((S - S4) ** 2).sum())
rhs = float((s[4:] ** 4).sum())
froS2 = float((S * S).sum())
assert abs(lhs - rhs) < 1e-9
assert abs(froS2 - (s**4).sum()) < 1e-9
assert np.isclose(lhs, 2.0758, atol=1e-4, rtol=0)
assert np.isclose(froS2, 28.6358, atol=1e-4, rtol=0)
print("rank-4 squared error, direct / spectrum:", lhs, rhs)
print("squared Frobenius norm of S:", froS2)

rank-4 squared error, direct / spectrum: 2.075766569930737 2.075766569930735
squared Frobenius norm of S: 28.635834215550222


### Answer check

In [2]:
assert W.shape == (12, 100)
assert W.dtype == np.float64
assert abs(lhs - rhs) < 1e-9
assert abs(froS2 - (s**4).sum()) < 1e-9
assert np.isclose(lhs, 2.0758, atol=1e-4, rtol=0)
assert np.isclose(froS2, 28.6358, atol=1e-4, rtol=0)